# Fine-tune LLM with Ray Core Distributed, PyTorch FSDP and QLora on Amazon SageMaker AI using ModelTrainer

In this notebook, we fine-tune LLM on Amazon SageMaker AI, using Python scripts and SageMaker ModelTrainer for executing a training job.

## Prerequisites

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade

In [ ]:
# Copy Ray launcher script to the scripts directory. 
%cp ../../../scripts/launcher.py ./scripts/

***

## Setup Configuration file path

In [ ]:
import os

# os.environ["AWS_PROFILE"] = "<aws_profile>"

In [ ]:
import os

model_id = "Qwen/Qwen3-0.6B"

os.environ["model_id"] = model_id

***

## Prepare the dataset

We are going to load [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT) dataset

In [ ]:
from sagemaker.core.helper.session_helper import get_execution_role, Session

In [ ]:
sagemaker_session = Session()
bucket_name = sagemaker_session.default_bucket()
region = sagemaker_session.boto_session.region_name
role = get_execution_role()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train[:10000]"
)

dataset

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.1, random_state=42)

print("Number of train elements: ", len(train))
print("Number of val elements: ", len(val))

Create a prompt template and load the dataset with a random sample to try summarization.

In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(model_id)

def prepare_dataset(sample):

    system_text = (
        "You are a deep-thinking AI assistant.\n\n"
        "For every user question, first write your thoughts and reasoning inside <think>...</think> tags, then provide your answer."
    )

    messages = []

    messages.append({"role": "system", "content": system_text})
    messages.append({"role": "user", "content": sample["Question"]})
    messages.append(
        {
            "role": "assistant",
            "content": f"<think>\n{sample['Complex_CoT'].lower()}\n</think>\n\n{sample['Response']}",
        }
    )

    # Apply chat template
    sample["text"] = tokenizer.apply_chat_template(messages, tokenize=False)

    return sample

In [ ]:
from datasets import Dataset, DatasetDict
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)

dataset = DatasetDict({"train": train_dataset, "val": val_dataset})

train_dataset = dataset["train"].map(
    prepare_dataset, remove_columns=list(train_dataset.features)
)

print(train_dataset[randint(0, len(dataset))]["text"])

val_dataset = dataset["val"].map(
    prepare_dataset, remove_columns=list(val_dataset.features)
)

### Upload to Amazon S3

In [ ]:
import boto3
import shutil
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()
s3_client = boto3.client("s3")

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
# save train_dataset to s3 using our SageMaker session
if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-modeltrainer-sft-ray"
else:
    input_path = f"datasets/llm-fine-tuning-modeltrainer-sft-ray"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.json"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.json"

In [ ]:
# Save datasets to s3
# We will fine tune only with 20 records due to limited compute resource for the workshop
train_dataset.to_json("./data/train/dataset.json", orient="records")
val_dataset.to_json("./data/val/dataset.json", orient="records")

s3_client.upload_file(
    "./data/train/dataset.json", bucket_name, f"{input_path}/train/dataset.json"
)
s3_client.upload_file(
    "./data/val/dataset.json", bucket_name, f"{input_path}/val/dataset.json"
)

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)

***

## (Optional) Copy Prometheus binary

In case you want to avoid Ray to download prometheus, you can copy the binary on S3 and pass as parameter to the Training job

In [ ]:
! wget https://github.com/prometheus/prometheus/releases/download/v3.13.1/prometheus-3.13.1.linux-amd64.tar.gz

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()
s3_client = boto3.client('s3')

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-modeltrainer-sft-ray"
else:
    input_path = f"datasets/llm-fine-tuning-modeltrainer-sft-ray"

prometheus_s3_path = (
    f"s3://{bucket_name}/{input_path}/prometheus/prometheus-3.13.1.linux-amd64.tar.gz"
)

In [ ]:
s3_client.upload_file(
    "./prometheus-3.13.1.linux-amd64.tar.gz",
    bucket_name,
    f"{input_path}/prometheus/prometheus-3.13.1.linux-amd64.tar.gz",
)

print(f"Prometheus binary uploaded to:")
print(prometheus_s3_path)

***

## Model fine-tuning

We are now ready to fine-tune our model. We will use the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) from transfomers to fine-tune our model. We prepared a script [train.py](./scripts/train.py) which will loads the dataset from disk, prepare the model, tokenizer and start the training.

For configuration we use `TrlParser`, that allows us to provide hyperparameters in a `yaml` file. This yaml will be uploaded and provided to Amazon SageMaker similar to our datasets. We are saving the config file as `args.yaml` and upload it to S3.

In [ ]:
%%bash

cat > ./args.yaml <<EOF
model_id: "${model_id}"                           # Hugging Face model id
# sagemaker specific parameters
output_dir: "/opt/ml/model"                       # path to where SageMaker will upload the model 
checkpoint_dir: "/opt/ml/checkpoints/"            # directory for saving training checkpoints
train_dataset_path: "/opt/ml/input/data/train/"   # path to where S3 saves train dataset
val_dataset_path: "/opt/ml/input/data/val/"       # path to where S3 saves test dataset
token: "${HF_TOKEN}"                              # Hugging Face API token; resolves to an empty string (and is then ignored) unless HF_TOKEN is exported in this notebook environment. Only needed for gated/private models.
merge_weights: true                               # merge weights in the base model
use_snapshot_download: false                      # use snapshot_download to download the model
# SFT specific parameters
max_grad_norm: 5.00e-01
apply_truncation: true                           # apply truncation to datasets
attn_implementation: "flash_attention_2"         # attention implementation type
learning_rate: 4.20e-04                          # learning rate scheduler
num_train_epochs: 2                              # number of training epochs
per_device_train_batch_size: 2                   # batch size per device during training
per_device_eval_batch_size: 2                    # batch size for evaluation
gradient_accumulation_steps: 2                   # number of steps before performing a backward/update pass
gradient_checkpointing: true                     # use gradient checkpointing
torch_dtype: "bfloat16"                          # float precision type
bf16: true                                       # use bfloat16 precision
tf32: true                                       # use tf32 precision
ignore_data_skip: true                           # skip data loading errors
logging_strategy: "steps"                        # logging strategy
logging_steps: 1                                 # log every N steps
log_on_each_node: false                          # disable logging on each node
ddp_find_unused_parameters: false                # DDP unused parameter detection
save_total_limit: 1                              # maximum number of checkpoints to keep
save_steps: 45                                   # Save checkpoint every this many steps
warmup_steps: 6                                  # number of warmup steps
weight_decay: 5.00e-02                           # weight decay coefficient
early_stopping: true                             # early stopping in case of decrease of eval loss
eval_strategy: "steps"                           # Add evaluation
eval_steps: 45                                   # Evaluate every ~half epoch
fsdp: "full_shard auto_wrap"                      # FSDP sharding strategy (no offload)
fsdp_config:                                     # FSDP configuration options
    transformer_layer_cls_to_wrap: "Qwen3DecoderLayer"  # Qwen3-0.6B decoder block class (matches model_id)
    backward_prefetch: "backward_pre"            # prefetch parameters during backward pass
    cpu_ram_efficient_loading: true              # enable CPU RAM efficient model loading
    offload_params: false                        # keep params/grads on GPU (see note above)
    forward_prefetch: false                      # disable forward prefetch
    use_orig_params: true                        # use original parameter names
# LoRA parameters
load_in_4bit: false                              # enable 4-bit quantization
lora_r: 16                                       # LoRA rank
lora_alpha: 32                                   # LoRA alpha parameter
lora_dropout: 0.1                                # LoRA dropout rate
EOF

Lets upload the config file to S3.

In [ ]:
import os

if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-modeltrainer-sft-ray"
else:
    input_path = f"datasets/llm-fine-tuning-modeltrainer-sft-ray"

train_config_s3_path = f"s3://{bucket_name}/{input_path}/config/args.yaml"

# upload the model yaml file to s3
model_yaml = "args.yaml"
s3_client.upload_file(model_yaml, bucket_name, f"{input_path}/config/args.yaml")
os.remove("./args.yaml")

print(f"Training config uploaded to:")
print(train_config_s3_path)

## Fine-tune model

Below estimtor will train the model with QLoRA, merge the adapter in the base model and save in S3

#### Get PyTorch image_uri

We are going to use the native PyTorch container image, pre-built for Amazon SageMaker

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
from sagemaker.train.configs import InstanceGroup

instance_groups = [
    InstanceGroup(
        instance_group_name="head-instance-group",
        instance_type="ml.t3.2xlarge",
        instance_count=1,
    ),
    InstanceGroup(
        instance_group_name="worker-instance-group",
        instance_type="ml.g5.xlarge",
        instance_count=4,
    ),
]

instance_groups

In [ ]:
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.8.0",
    instance_type=instance_groups[-1].instance_type,
    image_scope="training",
)

image_uri

In [ ]:
from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    RemoteDebugConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

args = [
    "--entrypoint",
    "train_ray.py",
    "--config",
    "/opt/ml/input/data/config/args.yaml",  # path to TRL config which was uploaded to s3
    # "--prometheus-path", # Enable these parameters in case of prometheus binary passed as InputData
    # "/opt/ml/input/data/prometheus/prometheus-3.13.1.linux-amd64.tar.gz", # Enable these parameters in case of prometheus binary passed as InputData
]

# Define the script to be run
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    command=f"python launcher.py {' '.join(args)}",
)

# Define the compute
compute_configs = Compute(
    instance_groups=instance_groups,
)

# define Training Job Name
job_name = f"train-{model_id.split('/')[-1].replace('.', '-')}-sft-ray"

# define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=18000),
    output_data_config=OutputDataConfig(
        s3_output_path=output_path, compression_type="NONE"
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoint", local_path="/opt/ml/checkpoints"
    ),
    environment={
        "head_instance_group": "head-instance-group",
        "head_num_cpus": "0",
        "head_num_gpus": "0",
        # Ray Train expects every node to see the same storage_path. This
        # cluster is multi-node (1 head + 4 workers), so /opt/ml/output/data
        # (node-local) would hide worker checkpoints from the driver.
        "RAY_STORAGE_PATH": output_path + "/ray-train",
        # "launch_prometheus": "false", # disable for local prometheus
        # "RAY_PROMETHEUS_HOST": "<PROMETHEUS_HOST>", # URL for remote prometheus server
        # "RAY_PROMETHEUS_NAME": "prometheus",
    },
    role=role,
).with_remote_debug_config(RemoteDebugConfig(enable_remote_debug=True))

In [ ]:
from sagemaker.train.configs import InputData, S3DataSource

# Pass the input data
train_input = InputData(
    channel_name="train",
    data_source=S3DataSource(
        s3_data_type="S3Prefix",
        s3_uri=train_dataset_s3_path,
        s3_data_distribution_type="FullyReplicated",
    ),  # S3 path where training data is stored
)

val_input = InputData(
    channel_name="val",
    data_source=S3DataSource(
        s3_data_type="S3Prefix",
        s3_uri=val_dataset_s3_path,
        s3_data_distribution_type="FullyReplicated",
    ),  # S3 path where val data is stored
)

config_input = InputData(
    channel_name="config",
    data_source=S3DataSource(
        s3_data_type="S3Prefix",
        s3_uri=train_config_s3_path,
        s3_data_distribution_type="FullyReplicated",
    ),  # S3 path where configs are stored
)

## Uncomment this lines if you want to provide the prometheus binary

# prometheus_input = InputData(
#     channel_name="prometheus",
#     data_source=S3DataSource(
#         s3_data_type="S3Prefix",
#         s3_uri=prometheus_s3_path,
#         s3_data_distribution_type="FullyReplicated",
#     ),  # S3 path where prometheus_s3_path binary is stored
# )

data = [
    train_input,
    val_input,
    config_input,
    # prometheus_input,
]

data

In [ ]:
# starting the train job with our uploaded datasets as input
model_trainer.train(input_data_config=data, wait=False)